# TrendPulse – What's Actually Trending Right Now

## Part 1: Python Foundations & Data Structures

**Project:** YouTube Trending Video Analysis

**Countries:** United States (US), Great Britain (GB), India (IN)

**Objective:** To load, clean, organize and analyze trending YouTube video records using core Python concepts, data structures, loops and rule-based classification.

In [6]:

import pandas as pd

countries = ["US", "GB", "IN"]

for country in countries:
    path = f"data/{country}videos.csv"
    df = pd.read_csv(path, nrows=5)

    print(f"\n{'=' * 40}")
    print(f"COUNTRY: {country}")
    print("Columns:")
    print(df.columns.tolist())

    print("\nTrending date examples:")
    print(df["trending_date"].head().tolist())

    print("\nPublish time examples:")
    print(df["publish_time"].head().tolist())

    print("\nFirst row:")
    print(df.head(1))

print("\nCATEGORY FILE")
categories_df = pd.read_csv("data/categories.csv", nrows=5)
print("Category columns:", categories_df.columns.tolist())
print(categories_df.head())



COUNTRY: US
Columns:
['video_id', 'trending_date', 'title', 'channel_title', 'category_id', 'publish_time', 'tags', 'views', 'likes', 'dislikes', 'comment_count', 'thumbnail_link', 'comments_disabled', 'ratings_disabled', 'video_error_or_removed', 'description']

Trending date examples:
['17.14.11', '17.14.11', '17.14.11', '17.14.11', '17.14.11']

Publish time examples:
['2017-11-14T07:04:28.000Z', '2017-11-12T22:35:48.000Z', '2017-11-13T08:40:22.000Z', '2017-10-27T22:21:47.000Z', '2017-11-09T13:20:20.000Z']

First row:
      video_id trending_date                         title  channel_title  \
0  g_N-w8LeAi8      17.14.11  Honest Football Prank LEAKED  QuietSpeech10   

   category_id              publish_time            tags  views   likes  \
0           22  2017-11-14T07:04:28.000Z  Monsoon|Robots  48555  2189.0   

   dislikes  comment_count                                  thumbnail_link  \
0       180          136.0  https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg   

   comment

In [7]:

import pandas as pd
from datetime import datetime

# 1. Configuration
DATA_DIR = "data"

COUNTRIES = ["US", "GB", "IN"]

VIRAL_VIEWS_THRESHOLD = 1_000_000
STEADY_VIEWS_THRESHOLD = 100_000
DIVISIVE_VIEWS_THRESHOLD = 50_000

HIGH_LIKE_RATIO = 0.70
LOW_LIKE_RATIO = 0.50

REQUIRED_COLUMNS = [
    "video_id", "trending_date", "title", "channel_title",
    "category_id", "publish_time", "views", "likes",
    "dislikes", "comment_count", "comments_disabled",
    "ratings_disabled", "video_error_or_removed"
]

# 2. Run header
start_time = datetime.now().astimezone()

print("=" * 50)
print("TREND PULSE - PART 1")
print(f"Countries to load: {', '.join(COUNTRIES)}")
print(f"Run started at: {start_time.strftime('%Y-%m-%d %H:%M:%S %Z')}")
print("=" * 50)


# 3. Reusable loading function
def load_country(path, country, max_rows=None):

    df = pd.read_csv(path, nrows=max_rows)

    missing_columns = [
        column for column in REQUIRED_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{country} file is missing columns: {missing_columns}"
        )

    df["country"] = country

    records = df.to_dict("records")

    print(f"{country}: {len(records):,} records loaded")

    return records


# 4. Load all three countries
all_records = []

for country in COUNTRIES:
    file_path = f"{DATA_DIR}/{country}videos.csv"

    country_records = load_country(file_path, country)

    all_records.extend(country_records)


# 5. Build category lookup dictionary
category_df = pd.read_csv(f"{DATA_DIR}/categories.csv")
category_records = category_df.to_dict("records")

category_names = {
    int(row["category_id"]): row["category_name"]
    for row in category_records
}

# 6. Loading summary
print("-" * 50)
print(f"Total records loaded: {len(all_records):,}")
print(f"Total categories: {len(category_names)}")

if len(all_records) >= 40_000:
    print("Requirement satisfied: At least 40,000 records loaded.")
else:
    print("WARNING: Total records are below 40,000.")

print("=" * 50)


TREND PULSE - PART 1
Countries to load: US, GB, IN
Run started at: 2026-11-12 09:57:38 UTC
US: 16,080 records loaded
GB: 15,075 records loaded
IN: 15,075 records loaded
--------------------------------------------------
Total records loaded: 46,230
Total categories: 17
Requirement satisfied: At least 40,000 records loaded.


In [17]:

# Numeric fields and their substitute values
NUMERIC_FIELDS = [
    "views",
    "likes",
    "dislikes",
    "comment_count"
]

# Boolean fields
BOOLEAN_FIELDS = [
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed"
]

# Count failed conversions
cleaning_failures = {
    field: 0 for field in NUMERIC_FIELDS + BOOLEAN_FIELDS
}

cleaning_failures["trending_date"] = 0
cleaning_failures["publish_time"] = 0


# Convert numeric values safely
def safe_int(value):
    try:
        if pd.isna(value) or str(value).strip().lower() in ["", "nan", "none"]:
            return 0, True

        number = float(value)

        if not number.is_integer():
            return 0, True

        return int(number), False

    except (ValueError, TypeError, OverflowError):
        return 0, True


# Convert Boolean values safely
def safe_bool(value):
    try:
        if pd.isna(value):
            return False, True

        if isinstance(value, bool):
            return value, False

        text = str(value).strip().lower()

        if text in ["true", "1", "yes"]:
            return True, False

        if text in ["false", "0", "no"]:
            return False, False

        return False, True

    except (ValueError, TypeError):
        return False, True


# Convert trending date: yy.dd.mm
def parse_trending_date(value):
    try:
        return datetime.strptime(
            str(value).strip(), "%y.%d.%m"
        ).date(), False

    except (ValueError, TypeError):
        return None, True


# Convert publish time: ISO datetime
def parse_publish_time(value):
    try:
        text = str(value).strip()

        if not text or text.lower() in ["nan", "none"]:
            return None, True

        return datetime.fromisoformat(
            text.replace("Z", "+00:00")
        ), False

    except (ValueError, TypeError):
        return None, True


# Clean all records
cleaned_records = []

for original_record in all_records:

    record = original_record.copy()

    # Clean numeric fields
    for field in NUMERIC_FIELDS:
        record[field], failed = safe_int(record[field])

        if failed:
            cleaning_failures[field] += 1

    # Clean Boolean fields
    for field in BOOLEAN_FIELDS:
        record[field], failed = safe_bool(record[field])

        if failed:
            cleaning_failures[field] += 1

    # Clean trending date
    record["trending_date"], failed = parse_trending_date(
        record["trending_date"]
    )

    if failed:
        cleaning_failures["trending_date"] += 1

    # Clean publish time
    record["publish_time"], failed = parse_publish_time(
        record["publish_time"]
    )

    if failed:
        cleaning_failures["publish_time"] += 1

    cleaned_records.append(record)


# Display cleaning summary
print("=" * 50)
print("DATA CLEANING SUMMARY")
print("=" * 50)

for field, count in cleaning_failures.items():
    print(f"{field}: {count} failed conversions")

print("-" * 50)
print(f"Original records: {len(all_records):,}")
print(f"Cleaned records: {len(cleaned_records):,}")

print("\nSample cleaned record:")
print(cleaned_records[0])

print("=" * 50)


DATA CLEANING SUMMARY
views: 0 failed conversions
likes: 12 failed conversions
dislikes: 0 failed conversions
comment_count: 36 failed conversions
comments_disabled: 0 failed conversions
ratings_disabled: 0 failed conversions
video_error_or_removed: 0 failed conversions
trending_date: 0 failed conversions
publish_time: 0 failed conversions
--------------------------------------------------
Original records: 46,230
Cleaned records: 46,230

Sample cleaned record:
{'video_id': 'g_N-w8LeAi8', 'trending_date': datetime.date(2017, 11, 14), 'title': 'Honest Football Prank LEAKED', 'channel_title': 'QuietSpeech10', 'category_id': 22, 'publish_time': datetime.datetime(2017, 11, 14, 7, 4, 28, tzinfo=datetime.timezone.utc), 'tags': 'Monsoon|Robots', 'views': 48555, 'likes': 2189, 'dislikes': 180, 'comment_count': 136, 'thumbnail_link': 'https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg', 'comments_disabled': False, 'ratings_disabled': False, 'video_error_or_removed': False, 'description': 'Watch thi


## Core Data Structures

- **List of dictionaries:** Stores all cleaned video records. Lists and dictionaries are mutable, so records can be updated when needed.
- **Tuple:** Stores important video fields such as video ID, trending date, views and likes. Tuples are immutable, so these values cannot be changed accidentally.
- **Set:** Finds unique video IDs and compares videos across countries without duplicates. Sets are mutable.
- **Dictionary:** Counts how many times each channel appears. Dictionaries are mutable.
- **Nested dictionary:** Organizes record counts by country and category. Dictionaries are mutable.


In [9]:

# 1. List of dictionaries
# cleaned_records already contains all cleaned records

print("Total cleaned records:", len(cleaned_records))


# 2. Tuple of important video fields
video_tuples = [
    (
        record["video_id"],
        record["trending_date"],
        record["views"],
        record["likes"]
    )
    for record in cleaned_records
]

print("\nSample video tuples:")
for item in video_tuples[:5]:
    print(item)


# 3. Sets: unique video IDs for each country
country_video_ids = {}

for country in COUNTRIES:
    country_video_ids[country] = {
        record["video_id"]
        for record in cleaned_records
        if record["country"] == country
    }

print("\nUnique video IDs per country:")

for country, video_ids in country_video_ids.items():
    print(f"{country}: {len(video_ids):,}")


# 4. Set intersection: videos appearing in all three countries
common_video_ids = (
    country_video_ids["US"]
    & country_video_ids["GB"]
    & country_video_ids["IN"]
)

print("\nVideos trending in all three countries:")
print(f"Count: {len(common_video_ids):,}")


# 5. Set difference: videos unique to each country
for country in COUNTRIES:

    other_country_ids = set()

    for other_country in COUNTRIES:
        if other_country != country:
            other_country_ids.update(
                country_video_ids[other_country]
            )

    unique_ids = country_video_ids[country] - other_country_ids

    print(
        f"Videos unique to {country}: {len(unique_ids):,}"
    )


# 6. Dictionary: count channel appearances
channel_counts = {}

for record in cleaned_records:
    channel = record["channel_title"]

    channel_counts[channel] = (
        channel_counts.get(channel, 0) + 1
    )


# Sort channels by appearance count
sorted_channels = sorted(
    channel_counts.items(),
    key=lambda item: item[1],
    reverse=True
)

# Slicing: top 15 channels
top_15_channels = sorted_channels[:15]

print("\nTop 15 channels by trending appearances:")

for rank, (channel, count) in enumerate(top_15_channels, start=1):
    print(f"{rank}. {channel}: {count:,}")


# 7. Nested dictionary: country -> category -> record count
country_category_summary = {}

for country in COUNTRIES:
    country_category_summary[country] = {}

    for record in cleaned_records:
        if record["country"] == country:

            category_id = record["category_id"]

            category_name = category_names.get(
                category_id, "Unknown"
            )

            country_category_summary[country][category_name] = (
                country_category_summary[country].get(category_name, 0) + 1
            )


# Display category summary
print("\nCountry-wise category summary:")

for country, categories in country_category_summary.items():

    print(f"\nCountry: {country}")

    for category, count in list(categories.items())[:5]:
        print(f"{category}: {count:,}")


print("\nCore data structures completed successfully!")


Total cleaned records: 46230

Sample video tuples:
('g_N-w8LeAi8', datetime.date(2017, 11, 14), 48555, 2189)
('Ugyql2NlPAd', datetime.date(2017, 11, 14), 198385, 1159)
('8fE19jkxJ9n', datetime.date(2017, 11, 14), 231656, 8742)
('WOY0nn7HCsO', datetime.date(2017, 11, 14), 8083, 176)
('2Lc_Cs5VVr_', datetime.date(2017, 11, 14), 373161, 11305)

Unique video IDs per country:
US: 5,068
GB: 4,113
IN: 5,878

Videos trending in all three countries:
Count: 20
Videos unique to US: 4,834
Videos unique to GB: 3,873
Videos unique to IN: 5,652

Top 15 channels by trending appearances:
1. QuietChallenge75: 2,464
2. OfficialReview41: 1,586
3. UltimateUnboxing65: 1,433
4. InsaneReaction46: 1,233
5. Brand NewChallenge45: 1,175
6. CrazyTrailer91: 1,018
7. FinalTrailer17: 911
8. UltimateSpeech20: 770
9. Brand NewSecret51: 684
10. WildHighlights7: 682
11. SurprisingHighlights41: 650
12. FinalBreakdown11: 643
13. CrazyRecipe67: 603
14. QuietInterview9: 570
15. UltimateVlog96: 565

Country-wise category summ

In [ ]:

## Variable Scope

A global variable is defined outside a function and can be accessed inside the function.

A local variable is defined inside a function and can only be accessed within that function.

In Python, local variables cannot normally be accessed directly outside their function.


In [10]:

# Global variable
project_name = "TrendPulse"


def scope_demo():

    # Local variable
    country_count = len(COUNTRIES)

    print("Inside function:")
    print(f"Global variable: {project_name}")
    print(f"Local variable: {country_count}")


# Call the function
scope_demo()


# Access global variable outside function
print("\nOutside function:")
print(f"Global variable: {project_name}")


# Try to access local variable outside function
try:
    print(country_count)

except NameError:
    print("Local variable cannot be accessed outside the function.")


Inside function:
Global variable: TrendPulse
Local variable: 3

Outside function:
Global variable: TrendPulse
Local variable cannot be accessed outside the function.



## Loops over Strings and Sequences

The title_features() function analyzes video titles by counting uppercase words, exclamation marks, question marks and total words.

A for loop processes characters and words. The range() function provides index-based access. A while loop scans video records until a high-view video is found. The break statement stops the loop, while continue skips records that do not meet the required condition.


In [11]:

# 1. Function to analyze video titles
def title_features(title):

    title = str(title)

    uppercase_words = 0

    # For loop: count uppercase words
    words = title.split()

    for word in words:
        if word.isupper() and any(char.isalpha() for char in word):
            uppercase_words += 1

    # Character loop: count punctuation marks
    exclamation_marks = 0
    question_marks = 0

    for character in title:
        if character == "!":
            exclamation_marks += 1

        elif character == "?":
            question_marks += 1

    # range() with index access
    indexed_characters = []

    for index in range(len(title)):
        indexed_characters.append(title[index])

    return {
        "uppercase_words": uppercase_words,
        "exclamation_marks": exclamation_marks,
        "question_marks": question_marks,
        "word_count": len(words)
    }


# 2. Test title_features()
sample_title = "WOW! AMAZING Video? Really COOL!"

print("Title:", sample_title)
print("Title features:", title_features(sample_title))


# 3. While loop with continue and break
VIEW_THRESHOLD = VIRAL_VIEWS_THRESHOLD

found_video = None
index = 0

while index < len(cleaned_records):

    record = cleaned_records[index]

    # Skip videos with zero or negative views
    if record["views"] <= 0:
        index += 1
        continue

    # Find the first video above the threshold
    if record["views"] > VIEW_THRESHOLD:
        found_video = record
        break

    index += 1


# 4. Display result
if found_video is not None:

    print("\nFirst video above the view threshold:")
    print(f"Title: {found_video['title']}")
    print(f"Country: {found_video['country']}")
    print(f"Views: {found_video['views']:,}")

else:
    print("\nNo video found above the threshold.")


print("\nLoop concepts completed successfully!")


Title: WOW! AMAZING Video? Really COOL!
Title features: {'uppercase_words': 3, 'exclamation_marks': 2, 'question_marks': 1, 'word_count': 5}

First video above the view threshold:
Title: Official Cats Speech EXCLUSIVE
Country: US
Views: 2,841,278

Loop concepts completed successfully!



## Rule-Based Video Classification

The project uses transparent rule-based thresholds:

- Viral Hit: Views >= 1,000,000 and like ratio >= 0.70.
- Divisive: Views >= 50,000 and like ratio < 0.50.
- Steady Performer: Views >= 100,000 and like ratio >= 0.50, with comments enabled and at least one comment, or comments disabled.
- Low Engagement: Videos that do not meet the above engagement conditions, including zero views.
- Unrated: Ratings are disabled or the total likes and dislikes are zero.

**Like ratio formula:**

like ratio = likes / (likes + dislikes)

**Edge cases:**
- Zero views are classified as Low Engagement.
- Zero total likes and dislikes are classified as Unrated.
- Ratings-disabled videos are classified as Unrated.
- Missing numeric values were replaced with zero during cleaning.
- When comments are disabled, comment count is not used to penalize the video.
- Invalid or negative engagement values are classified as Low Engagement.


In [12]:

# Classification thresholds
VIRAL_VIEWS_THRESHOLD = 1_000_000
STEADY_VIEWS_THRESHOLD = 100_000
DIVISIVE_VIEWS_THRESHOLD = 50_000

HIGH_LIKE_RATIO = 0.70
LOW_LIKE_RATIO = 0.50


def classify_video(record):

    views = record["views"]
    likes = record["likes"]
    dislikes = record["dislikes"]
    comments = record["comment_count"]

    comments_disabled = record["comments_disabled"]
    ratings_disabled = record["ratings_disabled"]

    # 1. Handle invalid or negative values
    if views < 0 or likes < 0 or dislikes < 0 or comments < 0:
        return "Low Engagement"

    # 2. Handle ratings-disabled videos
    if ratings_disabled:
        return "Unrated"

    # 3. Handle zero views
    if views == 0:
        return "Low Engagement"

    # 4. Calculate like ratio safely
    total_ratings = likes + dislikes

    if total_ratings == 0:
        return "Unrated"

    like_ratio = likes / total_ratings

    # 5. Nested classification conditions
    if views >= VIRAL_VIEWS_THRESHOLD:

        if like_ratio >= HIGH_LIKE_RATIO:
            return "Viral Hit"

        elif like_ratio < LOW_LIKE_RATIO:
            return "Divisive"

        else:
            return "Steady Performer"

    elif views >= STEADY_VIEWS_THRESHOLD:

        if like_ratio >= LOW_LIKE_RATIO:

            if comments_disabled or comments > 0:
                return "Steady Performer"

            else:
                return "Low Engagement"

        else:
            return "Divisive"

    elif views >= DIVISIVE_VIEWS_THRESHOLD:

        if like_ratio < LOW_LIKE_RATIO:
            return "Divisive"

        else:
            return "Low Engagement"

    else:
        return "Low Engagement"


# 6. Hand-trace example
example_record = {
    "views": 2_000_000,
    "likes": 90_000,
    "dislikes": 10_000,
    "comment_count": 500,
    "comments_disabled": False,
    "ratings_disabled": False
}

example_ratio = (
    example_record["likes"] /
    (example_record["likes"] + example_record["dislikes"])
)

print("HAND-TRACE EXAMPLE")
print(f"Views: {example_record['views']:,}")
print(f"Like ratio: {example_ratio:.2f}")
print("Classification:", classify_video(example_record))


# 7. Classify all cleaned records
classification_counts = {
    "Viral Hit": 0,
    "Steady Performer": 0,
    "Divisive": 0,
    "Low Engagement": 0,
    "Unrated": 0
}

for record in cleaned_records:

    label = classify_video(record)

    record["classification"] = label

    classification_counts[label] += 1


# 8. Display classification summary
print("\nVIDEO CLASSIFICATION SUMMARY")
print("=" * 40)

for label, count in classification_counts.items():
    print(f"{label}: {count:,}")

print("-" * 40)
print(f"Total classified: {sum(classification_counts.values()):,}")
print("Classification completed successfully!")


HAND-TRACE EXAMPLE
Views: 2,000,000
Like ratio: 0.90
Classification: Viral Hit

VIDEO CLASSIFICATION SUMMARY
Viral Hit: 9,555
Steady Performer: 27,345
Divisive: 47
Low Engagement: 8,739
Unrated: 544
----------------------------------------
Total classified: 46,230
Classification completed successfully!



## Refactoring

I created a reusable function to count values in a selected record field. This reduces repeated counting code and makes the notebook easier to maintain. The same function can count countries, channels and classification labels.


In [13]:

# Reusable counting function
def count_field_values(records, field):

    counts = {}

    for record in records:

        value = record[field]

        counts[value] = counts.get(value, 0) + 1

    return counts


# Count records by country
country_counts = count_field_values(
    cleaned_records, "country"
)

print("Records by country:")

for country, count in country_counts.items():
    print(f"{country}: {count:,}")


# Count records by channel
refactored_channel_counts = count_field_values(
    cleaned_records, "channel_title"
)

top_15_refactored = sorted(
    refactored_channel_counts.items(),
    key=lambda item: item[1],
    reverse=True
)[:15]

print("\nTop 15 channels:")

for rank, (channel, count) in enumerate(
    top_15_refactored, start=1
):
    print(f"{rank}. {channel}: {count:,}")


# Count classification labels
refactored_classification_counts = count_field_values(
    cleaned_records, "classification"
)

print("\nClassification counts:")

for label, count in refactored_classification_counts.items():
    print(f"{label}: {count:,}")


print("\nRefactoring completed successfully!")


Records by country:
US: 16,080
GB: 15,075
IN: 15,075

Top 15 channels:
1. QuietChallenge75: 2,464
2. OfficialReview41: 1,586
3. UltimateUnboxing65: 1,433
4. InsaneReaction46: 1,233
5. Brand NewChallenge45: 1,175
6. CrazyTrailer91: 1,018
7. FinalTrailer17: 911
8. UltimateSpeech20: 770
9. Brand NewSecret51: 684
10. WildHighlights7: 682
11. SurprisingHighlights41: 650
12. FinalBreakdown11: 643
13. CrazyRecipe67: 603
14. QuietInterview9: 570
15. UltimateVlog96: 565

Classification counts:
Low Engagement: 8,739
Steady Performer: 27,345
Viral Hit: 9,555
Divisive: 47
Unrated: 544

Refactoring completed successfully!


In [16]:
import math
def is_missing(value):
    if value is None:
        return True

    try:
        return math.isnan(value)
    except (TypeError, ValueError):
        return False


def safe_int(value):
    try:
        if is_missing(value) or str(value).strip().lower() in ["", "nan", "none"]:
            return 0, True

        number = float(value)

        if not math.isfinite(number) or not number.is_integer():
            return 0, True

        return int(number), False

    except (ValueError, TypeError, OverflowError):
        return 0, True


def safe_bool(value):
    try:
        if is_missing(value):
            return False, True

        if isinstance(value, bool):
            return value, False

        text = str(value).strip().lower()

        if text in ["true", "1", "yes"]:
            return True, False

        if text in ["false", "0", "no"]:
            return False, False

        return False, True

    except (ValueError, TypeError):
        return False, True